# Exploratory analysis with SQL

All queries run on SQLite with IBM's `Spacex.csv`.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv").to_sql("SPACEXTBL", con, index=False)
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")


def q(sql):
    return pd.read_sql(sql, con)


q("SELECT COUNT(*) AS records FROM SPACEXTABLE")

,records
0,101


In [2]:
q("SELECT DISTINCT Launch_Site FROM SPACEXTABLE")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


In [3]:
q("SELECT Date, Booster_Version, Launch_Site, Customer, Landing_Outcome FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Booster_Version,Launch_Site,Customer,Landing_Outcome
0,2010-06-04,F9 v1.0 B0003,CCAFS LC-40,SpaceX,Failure (parachute)
1,2010-12-08,F9 v1.0 B0004,CCAFS LC-40,NASA (COTS) NRO,Failure (parachute)
2,2012-05-22,F9 v1.0 B0005,CCAFS LC-40,NASA (COTS),No attempt
3,2012-10-08,F9 v1.0 B0006,CCAFS LC-40,NASA (CRS),No attempt
4,2013-03-01,F9 v1.0 B0007,CCAFS LC-40,NASA (CRS),No attempt


In [4]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)'")

,Total_Payload_Mass
0,45596


In [5]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass FROM SPACEXTABLE WHERE Booster_Version LIKE 'F9 v1.1%'")

,Avg_Payload_Mass
0,2534.666667


In [6]:
q("SELECT MIN(Date) AS First_Ground_Landing FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)'")

,First_Ground_Landing
0,2015-12-22


In [7]:
q("SELECT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


In [8]:
q("SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE 'Failure' END AS Outcome, COUNT(*) AS Total FROM SPACEXTABLE GROUP BY Outcome")

,Outcome,Total
0,Failure,1
1,Success,100


In [9]:
q("SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)")

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


In [10]:
q("SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTABLE WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'")

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [11]:
q("""SELECT Landing_Outcome, COUNT(*) AS Total, RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome ORDER BY Total DESC""")

,Landing_Outcome,Total,Rank
0,No attempt,10,1
1,Success (drone ship),5,2
2,Failure (drone ship),5,2
3,Success (ground pad),3,4
4,Controlled (ocean),3,4
5,Uncontrolled (ocean),2,6
6,Failure (parachute),2,6
7,Precluded (drone ship),1,8
